#Step 1: Set up

In [0]:
from pyspark.sql import functions as F, Window

CATALOG = "turbofan"   # your catalog from T04
SCHEMA  = "cmapss"
RUL_CAP = 125          # parameter, documented above
STD_TOL = 0.01         # starting threshold, reviewed in Step 2

bronze = spark.table(f"{CATALOG}.{SCHEMA}.bronze_train_fd001")
print(bronze.count())

#Step 2: Find the near-constant columns (T10)

In [0]:
feature_cols = [c for c in bronze.columns if c.startswith(("setting_", "s_"))]

std_pdf = (bronze.agg(*[F.stddev(c).alias(c) for c in feature_cols])
                 .toPandas().T.rename(columns={0: "std"}).sort_values("std"))
display(round(std_pdf,2))

In [0]:
drop_cols = std_pdf[std_pdf["std"] < STD_TOL].index.tolist()
print(len(drop_cols), drop_cols)

## Silver layer: RUL labels and column cleaning (FD001)

**What this notebook does**
1. Reads the bronze train and test tables.
2. Adds `rul_raw` (last cycle of the engine minus the current cycle) and `rul`, the same value capped at 125.
3. Drops columns that carry no information about degradation.
4. Writes `silver_train_fd001` and `silver_test_fd001`.

**Columns dropped (10):** `setting_1`, `setting_2`, `setting_3`, `s_1`, `s_5`, `s_6`, `s_10`, `s_16`, `s_18`, `s_19`

**Why these were dropped**
- Their standard deviation across the training data was below 0.01, so they stay (almost) constant for the whole life of an engine. A column that never changes cannot help detect or predict failure.
- FD001 has a single operating condition, so the three operating settings are noise, not signal.
- The same sensors are removed in published C-MAPSS studies, which leaves 14 informative sensors.
- The drop list was decided from the **training data only** and applied unchanged to the test data, so no test information influences the cleaning.

**Why the RUL is capped at 125**
- Early in an engine's life there is no visible degradation, so the true remaining life cannot be read from the sensors.
- The common approach is a piece-wise linear target: constant at 125 early on, then falling linearly to 0 at failure.
- `rul_raw` is kept beside `rul` so the choice can be revisited at evaluation time.

**Result:** 14 sensors kept, one `rul_raw` and one capped `rul` column per row, and all validation checks passed (row counts match bronze, every engine ends at RUL 0, no values above the cap).

#Step 3: Compute RUL (T09) and apply the cap (T11)

In [0]:
w = Window.partitionBy("unit")

silver_train = (bronze
    .withColumn("rul_raw", F.max("cycle").over(w) - F.col("cycle"))   # last cycle - current cycle
    .withColumn("rul", F.least(F.col("rul_raw"), F.lit(RUL_CAP)))     # capped target
    .drop(*drop_cols))

silver_test = (spark.table(f"{CATALOG}.{SCHEMA}.bronze_test_fd001")
    .drop(*drop_cols))   # same columns dropped, decided from train only

#Step 4: Write the silver tables

In [0]:
silver_train.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_train_fd001")
silver_test.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_test_fd001")

#Step 5: Check the work

In [0]:
s  = spark.table(f"{CATALOG}.{SCHEMA}.silver_train_fd001")
st = spark.table(f"{CATALOG}.{SCHEMA}.silver_test_fd001")

print("rows silver vs bronze:", s.count(), bronze.count())                       # equal
print("engines whose last RUL is not 0:",
      s.groupBy("unit").agg(F.min("rul_raw").alias("m")).filter("m != 0").count())  # 0
print("rows above cap:", s.filter(F.col("rul") > RUL_CAP).count())                # 0
print("share of rows at the cap:", round(s.filter(F.col("rul") == RUL_CAP).count() / s.count(), 3))
print("columns match:", set(st.columns) == set(s.columns) - {"rul_raw", "rul"})   # True
display(s.filter("unit = 1").orderBy("cycle").select("unit", "cycle", "rul_raw", "rul").limit(5))

In [0]:
s.filter("unit = 1").agg(F.max("cycle"))

In [0]:
from pyspark.sql import functions as F

e1 = s.filter("unit = 1")

max_cycle     = e1.agg(F.max("cycle")).collect()[0][0]
first_rul_raw = e1.orderBy("cycle").select("rul_raw").first()[0]
last_rul_raw  = e1.orderBy(F.col("cycle").desc()).select("rul_raw").first()[0]

print("highest cycle for engine 1:", max_cycle)
print("rul_raw on cycle 1:", first_rul_raw, "| expected:", max_cycle - 1)
print("rul_raw on last cycle:", last_rul_raw, "| expected: 0")
print("check passed:", first_rul_raw == max_cycle - 1 and last_rul_raw == 0)